# 🦈 Orbit Wars: Apex Predator Strategy (v5)
**Kaggle Public Score: 608.2**

Unlike the Archangel agent, "Apex Predator" does not just make decisions based on the current state; it features a structure that **simulates time**. It transforms planets from simple data structures into `PlanetNode` objects with their own internal event queue system capable of predicting future events.

Its biggest innovations are: **Ghost Threat Detection** and **Dynamic Simulation Injection**.

## 1. Timeline Simulator (`PlanetNode`)
Classic agents only look at the instantaneous situation. Apex Predator, however, creates an arrival queue (`arrivals`) for each planet. It calculates when a fleet will arrive and uses `get_future_state(eta)` to simulate **who will own the planet and how many ships it will have** at that exact moment, factoring in production rates over time.

In [10]:
class PlanetNode:
    def __init__(self, data):
        self.id, self.owner, self.x, self.y, self.radius, self.ships, self.production = data
        self.raw_data = {'id': self.id, 'x': self.x, 'y': self.y, 'radius': self.radius}
        self.arrivals = [] 

    def add_arrival(self, eta, owner, ships):
        self.arrivals.append((eta, owner, ships))

    def get_future_state(self, target_eta):
        """Simulates who the planet will belong to after a specified time (eta)."""
        curr_owner, curr_ships = self.owner, self.ships
        self.arrivals.sort(key=lambda x: x[0])
        last_t = 0
        for arr_t, arr_owner, arr_ships in self.arrivals:
            if arr_t > target_eta: break
            dt = arr_t - last_t
            if curr_owner != -1: curr_ships += self.production * dt
            if arr_owner == curr_owner: curr_ships += arr_ships
            else:
                if arr_ships > curr_ships:
                    curr_owner, curr_ships = arr_owner, arr_ships - curr_ships
                else: curr_ships -= arr_ships
            last_t = arr_t
        dt = target_eta - last_t
        if curr_owner != -1: curr_ships += self.production * dt
        return curr_owner, curr_ships

## 2. Ghost Threat & Safe Pool
Even if an enemy hasn't launched a fleet yet, their close proximity is a hidden threat. Apex Predator measures the potential damage from surrounding enemies (`eta_threat`) and holds fleets in a safe reserve (`available_pool`) before an attack even occurs.

In [ ]:
# --- EXCERPT FROM MAIN AGENT LOOP: GHOST THREAT ---
# Note: Variables like my_planets, enemy_planets, and time_left are parsed dynamically from the 'obs' payload.

available_pool = {}
for src in my_planets:
    ghost_threat = 0
    closest_enemy_dist = 999.0
    for e in enemy_planets:
        dist = get_dist(src.x, src.y, e.x, e.y)
        if dist < closest_enemy_dist: closest_enemy_dist = dist
        eta_threat = dist / MAX_SPEED
        
        # Calculate potential threat if the enemy is closer than 12 turns
        if eta_threat < 12: 
            potential_dmg = e.ships - (src.ships + src.production * eta_threat)
            if potential_dmg > ghost_threat: ghost_threat = int(potential_dmg * 0.4) 
            
    safe_turns = closest_enemy_dist / MAX_SPEED
    buffer = 0 if safe_turns > 12 else int(src.production * 1.5) + ghost_threat
    
    req_def = src.get_required_defense(time_left, player)
    avail = int(src.ships - (req_def + buffer))
    available_pool[src.id] = max(0, avail)

## 3. Dynamic Simulation Injection & Smart Logistics
Many agents unnecessarily send multiple fleets to the same target (Over-committing). When Apex Predator decides to attack a target, it **immediately injects the arrival of its own fleet into that planet's simulation** (`planets[tid].add_arrival`). This way, our other planets know that the target is already scheduled to be captured and do not waste their fleets. 

Finally, the agent distributes its idle ships to the frontlines as logistical support (e.g., splitting a 60%-40% ratio across two separate fronts) rather than pooling them uselessly in the rear.

In [ ]:
# --- EXCERPT FROM MAIN AGENT LOOP: INJECTION & LOGISTICS ---

# 3. SWARM ATTACK & INJECTION
for move in candidate_moves:
    sid, tid, eta, tx, ty = move['src_id'], move['tgt_id'], move['eta'], move['tx'], move['ty']
    if available_pool[sid] <= 0: continue
    
    # Predict the target's future state using our internal timeline simulator
    current_future_owner, current_future_ships = planets[tid].get_future_state(eta)
    if current_future_owner == player: continue 
        
    real_req = max(1, int(current_future_ships + 2))
    
    if available_pool[sid] >= real_req:
        angle = math.atan2(ty - planets[sid].y, tx - planets[sid].x)
        actions.append([int(sid), float(angle), int(real_req)])
        
        available_pool[sid] -= real_req
        
        # INJECTION: Update the target's simulation immediately so other planets don't over-commit!
        planets[tid].add_arrival(eta, player, real_req) 

# 4. SMART LOGISTICS (Multi-Front Wall)
if (time.time() - start_time) < 0.85:
    for src in my_planets:
        if available_pool[src.id] > 30 and enemy_planets:
            frontlines = sorted([p for p in my_planets if p.id != src.id], 
                                key=lambda p: min([get_dist(p.x, p.y, e.x, e.y) for e in enemy_planets]))
            
            if frontlines:
                targets = [(frontlines[0], 1.0)]
                if len(frontlines) > 1:
                    # Split the support 60/40 between the top two frontlines
                    targets = [(frontlines[0], 0.6), (frontlines[1], 0.4)]
                    
                for f_planet, ratio in targets:
                    send_amount = int(available_pool[src.id] * ratio)
                    if send_amount < 5: continue
                    
                    angle = math.atan2(f_planet.y - src.y, f_planet.x - src.x)
                    if not path_hits_sun(src.x, src.y, f_planet.x, f_planet.y):
                        actions.append([int(src.id), float(angle), send_amount])